# 3 · Building with LangChain, RAG, and Tools
La misma aplicación recupera evidencia y ejecuta capacidades autorizadas.
La búsqueda local es léxica; no se presenta como embeddings ni semantic retrieval.

Predice el resultado antes de ejecutar y anota tus observaciones.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Cada notebook empieza desde datos preparados, sin archivos de sesiones anteriores.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv

if not os.getenv("CARRITO_NOTEBOOK_CHECK"):
    load_dotenv(ROOT / ".env")
from carrito.store import create_store
from carrito.tools import StoreTools

db = create_store()
tools = StoreTools(db, user_id="user1")  # Identidad fijada por el host.
RUN_LIVE = False  # Cambiar explícitamente a True permite llamadas de pago.
COMPLETION = {}

def check(name, condition):
    COMPLETION[name] = bool(condition)
    print(("OK" if condition else "PENDIENTE") + ": " + name)

print("MODO OFFLINE: fixtures deterministas. No miden calidad del LLM.")

## Documentos → unidades recuperables → resultados
Las políticas ya son unidades cortas: dividirlas arbitrariamente puede separar regla y excepción. Inspecciona metadata y decide qué frontera conservarías en un manual más largo.

In [ ]:
from carrito.retrieval import load_chunks, retrieval_report, retrieve

chunks = load_chunks()
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))
for query in ["devolución", "reintegro", "personalizados devolución"]:
    print(query, [(r["source_id"], r["score"]) for r in retrieve(query, chunks, top_k=2)])

## Calidad de retrieval
Predice fallos para sinónimos y excepciones. Compara top-k=1/2 con y sin expansión: cuatro condiciones, no dos cambios confundidos. Completa una configuración y conserva el detalle por consulta; mejorar recall puede empeorar precision. La expansión explícita no es comprensión semántica.

In [ ]:
def retrieval_candidate():
    # TODO: top_k=2 y expansión explícita; comparar después las cuatro condiciones.
    return retrieval_report(chunks, top_k=1, expand=False)

In [ ]:
baseline = retrieval_report(chunks, top_k=1, expand=False)
candidate = retrieval_candidate()
for k in [1, 2]:
    for expand in [False, True]:
        report = retrieval_report(chunks, top_k=k, expand=expand)
        print({"top_k": k, "expand": expand, "recall": report["mean_recall"], "precision": report["mean_precision"]})
print(json.dumps(candidate["cases"], ensure_ascii=False, indent=2))
check("mejora observada de recall", candidate["mean_recall"] > baseline["mean_recall"])
# Una cita recuperada no implica que su texto respalde la afirmación.
claims = [{"claim": "Toda compra se puede devolver", "cited_id": "POL-EXC", "supported": None}]
print("Revisión de grounding:", claims)

## Tool calling nativo
Antes de ejecutar, identifica call_id, nombre, arguments y resultado. Configura solo lectura; una petición de escritura debe rechazarse incluso si el modelo la propone. El mismo `run_profile` continúa en S4 y S5.

In [ ]:
from functools import partial

from carrito.lab import AgentProfile, run_profile
from carrito.model import FixtureModel

# La estrategia medida se usa ahora en la tool real del agente.
tools.policy_retriever = partial(retrieve, chunks=chunks, top_k=2, expand=True)
policy_call = {"type": "function_call", "name": "search_policies", "call_id": "policy", "arguments": json.dumps({"query": "reintegro"})}
policy_trace = run_profile("Explica reintegro", tools, FixtureModel([[policy_call], []]), AgentProfile())
print("Evidencia del agente:", [e["result"] for e in policy_trace["events"] if e["type"] == "tool_result"])
call = {"type": "function_call", "name": "get_order", "call_id": "order_104", "arguments": json.dumps({"order_id": "104"})}
profile = AgentProfile(name="order-reader", allowed_tools=("get_order",), enable_skills=False)
trace = run_profile("Consulta 104", tools, FixtureModel([[call], []], "[FIXTURE] Resultado observado"), profile)
print(json.dumps(trace, ensure_ascii=False, indent=2))

In [ ]:
def result_for_call(events, call_id):
    # TODO: localizar tool_result correspondiente al call_id.
    return None

In [ ]:
observed = result_for_call(trace["events"], "order_104")
check("call_id enlaza observación", observed == tools.get_order("104"))
for order_id in ["104", "109", "999"]:
    print(order_id, tools.dispatch("get_order", {"order_id": order_id}))
print("Argumento ajeno:", tools.dispatch("get_order", {"order_id": "104", "user_id": "user2"}))
print("Define expected behavior para una tool desconocida y para JSON malformado.")

## MCP: integración ejecutable
Host → cliente → proceso servidor. Inspecciona `examples/mcp/server.py`: el server expone catálogo, no permisos para devoluciones. Compara el mismo resultado local y remoto; usa el resultado MCP como evidencia para el siguiente request.

In [ ]:
from carrito.mcp_client import catalog_smoke

mcp_result = await catalog_smoke()
print(json.dumps(mcp_result, ensure_ascii=False, indent=2))
from carrito.context import ConversationState, compose_context

mcp_pack = compose_context("Recomienda", ConversationState(budget_eur=80), [{"source_id": "MCP-CATALOG", "text": mcp_result["content"], "relevant": True}], 10000)
print(mcp_pack["messages"])
assert mcp_result["transport"] == "stdio" and mcp_result["read_only"]

## LangChain: qué abstrae
Compara request/response nativos con `StructuredTool`, `AIMessage.tool_calls` y `ToolMessage`. La función y autorización siguen en Python. El ejemplo tiene dos llamadas y una capacidad de lectura; no es otro agente.

In [ ]:
from importlib.util import find_spec

if find_spec("langchain_openai"):
    from carrito.langchain_demo import run_comparison
    print(json.dumps(run_comparison(live=False), ensure_ascii=False, indent=2))
else:
    print("Antes de clase: uv sync --locked --extra langchain")

## Salida de sesión
Decide para cada necesidad: retrieval, tool, long context o ninguna. Documenta una consulta fallida, una cita no sustentada y una denegación correcta. S4 decide cuándo delegar la secuencia al modelo.

In [ ]:
print(json.dumps(COMPLETION, ensure_ascii=False, indent=2))
print("CHECKPOINT_COMPLETO" if all(COMPLETION.values()) else "Completa las celdas TODO y repite los checks.")